<a href="https://colab.research.google.com/github/camilavazquezz/colab/blob/main/Part%201%20house_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 1: House Price Prediction Using Linear Regression

This project uses Linear Regression to predict house prices based on square footage and location. Location is represented using Downtown, Rural, and Suburb categories and is processed using OneHotEncoder.

In [20]:
import kagglehub
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Data source: Kaggle - House Sales in King County, USA
# Dataset by harlfoxem
path = kagglehub.dataset_download("harlfoxem/housesalesprediction")

print("Dataset downloaded to:", path)
print("Files:", os.listdir(path))

Using Colab cache for faster access to the 'housesalesprediction' dataset.
Dataset downloaded to: /kaggle/input/housesalesprediction
Files: ['kc_house_data.csv']


## Dataset

This project uses the House Sales in King County, USA dataset from Kaggle. The dataset contains over 21,000 house sales with information about house prices, square footage, location, and other housing characteristics.

In [21]:
# Load the housing dataset
csv_path = os.path.join(path, "kc_house_data.csv")

df = pd.read_csv(csv_path)

df.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

Data source: https://www.kaggle.com/datasets/harlfoxem/housesalesprediction?resource=download


In [23]:
print("Number of houses:", len(df))
print("\nColumns:")
print(df.columns.tolist())

Number of houses: 21613

Columns:
['id', 'date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long', 'sqft_living15', 'sqft_lot15']


In [24]:
# Select house price and square footage from the real dataset
house_data = df[['sqft_living', 'price']].copy()

# Rename square footage to match the starter code
house_data = house_data.rename(columns={
    'sqft_living': 'square_footage'
})

# Generate location categories to match the starter code
# A fixed random seed makes the generated locations reproducible
np.random.seed(42)

house_data['location'] = np.random.choice(
    ['Downtown', 'Rural', 'Suburb'],
    size=len(house_data)
)

# Display the first 10 houses
house_data.head(10)

,square_footage,price,location
0,1180,221900.0,Suburb
1,2570,538000.0,Downtown
2,770,180000.0,Suburb
3,1960,604000.0,Suburb
4,1680,510000.0,Downtown
5,5420,1225000.0,Downtown
6,1715,257500.0,Suburb
7,1060,291850.0,Rural
8,1780,229500.0,Suburb
9,1890,323000.0,Suburb


In [25]:
# Features used to predict the house price
X = house_data[['square_footage', 'location']]

# Target variable (what we want to predict)
y = house_data['price']

print("Features:")
print(X.head())

print("\nTarget prices:")
print(y.head())

Features:
   square_footage  location
0            1180    Suburb
1            2570  Downtown
2             770    Suburb
3            1960    Suburb
4            1680  Downtown

Target prices:
0    221900.0
1    538000.0
2    180000.0
3    604000.0
4    510000.0
Name: price, dtype: float64


In [26]:
# Preprocess the location column using OneHotEncoder
preprocessor = ColumnTransformer(
    transformers=[
        ('location', OneHotEncoder(handle_unknown='ignore'), ['location'])
    ],
    remainder='passthrough'
)

# Create the Linear Regression model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Split the real housing data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

print("Model trained successfully!")
print("Training houses:", len(X_train))
print("Testing houses:", len(X_test))

Model trained successfully!
Training houses: 17290
Testing houses: 4323


In [27]:
# Predict the price of a 2000 sq ft house in Downtown
new_house = pd.DataFrame({
    'square_footage': [2000],
    'location': ['Downtown']
})

predicted_price = model.predict(new_house)

print(
    f"Predicted price for a 2000 sq ft house in Downtown: "
    f"${predicted_price[0]:,.2f}"
)

Predicted price for a 2000 sq ft house in ZIP code 98125: $517,100.22


In [28]:
# Get feature names and model coefficients
feature_names = model.named_steps['preprocessor'].get_feature_names_out()
coefficients = model.named_steps['regressor'].coef_

print("Model Coefficients:")

for feature, coef in zip(feature_names, coefficients):
    print(f"{feature}: ${coef:,.2f}")

Model Coefficients:
location__location_Downtown: $-6,137.09
location__location_Rural: $3,619.60
location__location_Suburb: $2,517.49
remainder__square_footage: $279.62


## Model Interpretation

The square footage coefficient is approximately **$279.62 per square foot**. This means that the model estimates that each additional square foot is associated with about a $279.62 increase in the predicted house price, while accounting for location.

Location also affects the predicted price. OneHotEncoder converts Downtown, Rural, and Suburb into numerical features so that the Linear Regression model can account for differences between location categories. The location coefficients show the location effects learned by the model.